# Dr.Copper — Phase 4: XGB and LSTM

In [1]:
import pandas as pd
import numpy as np

import dr_copper as dc
from pathlib import Path
import joblib

import warnings

warnings.filterwarnings("ignore")

FIGURES = Path("../outputs/figures")
DATA = Path("../data")

## 1 — Load previous Phases outputs

In [2]:
features = pd.read_parquet(DATA / "processed" / "features.parquet")
features_train = pd.read_parquet(DATA / "processed" / "features_train.parquet")
features_test = pd.read_parquet(DATA / "processed" / "features_test.parquet")

features_with_vol = pd.read_parquet(DATA / "processed" / "features_with_vol.parquet")

In [3]:
pca_pipeline = joblib.load(DATA / "processed" / "pca_pipeline.joblib")
pc_scores_train = pd.read_parquet(DATA / "processed" / "pc_scores.parquet")

In [4]:
km_model = joblib.load(DATA / "processed" / "km.joblib")
regime_labels_train = pd.read_parquet(DATA / "processed" / "regime_labels.parquet")

## 2 — Data preparation

In [5]:
features_with_vol_train = features_with_vol.loc[features_train.index]
features_with_vol_test = features_with_vol.loc[features_test.index]

### Train data

Concatenate the train data set with volatility with the prepared pc_scores and regime_labels for the train data.

In [6]:
train = pd.concat(
    [features_with_vol_train, pc_scores_train, regime_labels_train], axis=1
)

As regimes are [0, 1, 2, 3], they appear to be a sequence, but they are not. So better to use one-hot encoding for them.

In [7]:
ohe = pd.get_dummies(train["regime"], prefix="regime", dtype=int)

train = train.join(ohe).drop(columns=["regime"])

### Test data

For the test data, we haven't run PCA and k-means before, so we need to do it now.

In [8]:
pc_scores_test = dc.transform_pca(features_with_vol_test, pca_pipeline)

In [9]:
regime_labels_test = dc.predict_regimes(pc_scores_test, km_model)

regime_labels_test = regime_labels_test.to_frame()

Now we can concatenate data for the test period as well.

In [10]:
test = pd.concat([features_with_vol_test, pc_scores_test, regime_labels_test], axis=1)

ohe = pd.get_dummies(test["regime"], prefix="regime", dtype=int)

test = test.join(ohe).drop(columns=["regime"])

### Target

As a target, we want the direction of the log return for 5 days forward.
Since we used dropna() during feature engineering, the actual data sets don't have consecutive trading days. 
So we need to calculate them from raw data.

In [11]:
TARGET = "copper_ret5d_forward"

raw = pd.read_parquet(DATA / "raw" / "raw.parquet")[["copper"]]

log_prices = np.log(raw)

raw["copper_ret5d_raw"] = log_prices["copper"].diff(5)
raw[TARGET] = raw["copper_ret5d_raw"].shift(-5)

In [12]:
train = train.join(raw[TARGET])
test = test.join(raw[TARGET])

train.dropna(inplace=True)
test.dropna(inplace=True)

In [13]:
X_train = train.drop(columns=TARGET)
y_train = train[TARGET]

X_test = test.drop(columns=TARGET)
y_test = test[TARGET]

In [14]:
y_train = pd.Series(np.sign(y_train)).replace(0, np.nan).bfill()
y_test = pd.Series(np.sign(y_test)).replace(0, np.nan).bfill()

y_train = (y_train + 1) / 2
y_test = (y_test + 1) / 2

## 3 — LSTM

LSTM model demands a sequence, and we need to build it.

In [15]:
X_seq_train, y_seq_train, d_seq_train = dc.build_sequences(X_train, y_train, 180)
X_seq_test, y_seq_test, d_seq_test = dc.build_sequences(X_test, y_test, 180)

In [16]:
config_lstm = dc.LSTMConfig(
    input_size=X_seq_train.shape[2],
    max_epochs=100,
    patience=30,
    lr=1e-3,
    dropout=0.4,
    hidden_size=64,
    num_layers=2,
)
data_lstm = dc.LSTMTrainerData(
    X_seq_train=X_seq_train,
    y_seq_train=y_seq_train,
    d_seq_train=d_seq_train,
    X_seq_test=X_seq_test,
    y_seq_test=y_seq_test,
    d_seq_test=d_seq_test,
)

In [17]:
lstm_trainer = dc.CopperLSTMTrainer(cfg=config_lstm, data=data_lstm)

In [18]:
fit_model_lstm = lstm_trainer.train()

Epoch [10/100], train BCELoss: 0.6925, val BCELoss: 0.6970
Epoch [20/100], train BCELoss: 0.6887, val BCELoss: 0.6978
Epoch [30/100], train BCELoss: 0.6884, val BCELoss: 0.6922
Epoch [40/100], train BCELoss: 0.6879, val BCELoss: 0.6950
Epoch [50/100], train BCELoss: 0.6864, val BCELoss: 0.6970
Epoch [60/100], train BCELoss: 0.6850, val BCELoss: 0.6947
Epoch [70/100], train BCELoss: 0.6848, val BCELoss: 0.6926
Epoch [80/100], train BCELoss: 0.6828, val BCELoss: 0.6925
Training stopped on 80 epoch. The best epoch is 50.
Test BCELoss: 0.6868


In [19]:
dc.lstm_test_score(data=data_lstm, fit_model=fit_model_lstm)

Accuracy score: 55.20%
ROC AUC score: 52.26%


## 4 — XGBoost

We are not aiming for optimal performance from the boosting model, so we will use a simple implementation with reasonable parameters without any hyperparameter tuning.

In [20]:
config_xgb = dc.XGBConfig(
    n_estimators=50,
    max_depth=2,
    lr=0.15,
    subsample=0.7,
    colsample_bytree=0.5,
    reg_lambda=10,
    reg_alpha=5,
    min_child_weight=10,
)
data_xgb = dc.XGBTrainerData(
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test
)

In [21]:
xgb_trainer = dc.CopperXGBTrainer(cfg=config_xgb, data=data_xgb)

In [22]:
fit_model_xgb = xgb_trainer.train()

In [23]:
dc.xgb_test_score(data=data_xgb, fit_model=fit_model_xgb)

Accuracy score: 55.10%
ROC AUC score: 53.39%


## 5 — XGBoost-LSTM model based on feature fusion

LSTM captures temporal dependencies through sequential context, while XGBoost captures non-linear relationships in tabular macro features. Inspired by hybrid model approaches in the literature, we combine both by appending XGBoost predicted probabilities as an additional feature to the LSTM input, allowing the sequential model to condition on the tree-based signal at each timestep.

First, we need to take predicted probabilities from the XGB model for the train and test datasets.

In [24]:
fusion_feature_train = fit_model_xgb.predict_proba(X_train)[:, 1]
fusion_feature_test = fit_model_xgb.predict_proba(X_test)[:, 1]

In [25]:
X_train_ff = X_train.copy()
X_test_ff = X_test.copy()

In [26]:
X_train_ff["xgb_prob"] = fusion_feature_train
X_test_ff["xgb_prob"] = fusion_feature_test

Then we can build sequences and launch LSTM training once again with the fusion feature.

In [27]:
X_seq_train_ff, y_seq_train, d_seq_train = dc.build_sequences(X_train_ff, y_train, 180)
X_seq_test_ff, y_seq_test, d_seq_test = dc.build_sequences(X_test_ff, y_test, 180)

In [28]:
config_lstm_xgb = dc.LSTMConfig(
    input_size=X_seq_train_ff.shape[2],
    max_epochs=100,
    patience=30,
    lr=1e-3,
    dropout=0.4,
    hidden_size=64,
    num_layers=2,
)
data_lstm_xgb = dc.LSTMTrainerData(
    X_seq_train=X_seq_train_ff,
    y_seq_train=y_seq_train,
    d_seq_train=d_seq_train,
    X_seq_test=X_seq_test_ff,
    y_seq_test=y_seq_test,
    d_seq_test=d_seq_test,
)

lstm_xgb_trainer = dc.CopperLSTMTrainer(cfg=config_lstm_xgb, data=data_lstm_xgb)

In [29]:
fit_model_lstm_xgb = lstm_xgb_trainer.train()

Epoch [10/100], train BCELoss: 0.6911, val BCELoss: 0.6980
Epoch [20/100], train BCELoss: 0.6877, val BCELoss: 0.6933
Epoch [30/100], train BCELoss: 0.6902, val BCELoss: 0.6950
Epoch [40/100], train BCELoss: 0.6883, val BCELoss: 0.6926
Epoch [50/100], train BCELoss: 0.6877, val BCELoss: 0.6958
Epoch [60/100], train BCELoss: 0.6855, val BCELoss: 0.7052
Epoch [70/100], train BCELoss: 0.6792, val BCELoss: 0.6834
Epoch [80/100], train BCELoss: 0.6742, val BCELoss: 0.6765
Epoch [90/100], train BCELoss: 0.6543, val BCELoss: 0.6784
Epoch [100/100], train BCELoss: 0.6420, val BCELoss: 0.6838
Test BCELoss: 0.6922


In [30]:
dc.lstm_test_score(data=data_lstm_xgb, fit_model=fit_model_lstm_xgb)

Accuracy score: 60.00%
ROC AUC score: 56.18%


## 6 — Save pipeline artefacts

In [31]:
import joblib

joblib.dump(fit_model_lstm, DATA / "processed" / "lstm.joblib")
joblib.dump(fit_model_xgb, DATA / "processed" / "xgb.joblib")
joblib.dump(fit_model_lstm_xgb, DATA / "processed" / "lstm_xgb.joblib")

['..\\data\\processed\\lstm_xgb.joblib']